In [4]:
# Nombres y apellidos completos: Yuliana Orihuela Lazo
# Código de matrícula: 2024200514G
# Tema y número del temario: Perpetuidades y valuación de acciones con
#                             dividendo estable en la BVL - Tema 27
# Fecha de extracción: 24/09/2026
"""
04_analisis.py
Estimaciones, tablas y figuras del artículo. Toma el archivo procesado por
03_limpieza_datos.py y calcula el resultado central de la investigación:
Ke (CAPM), valor teórico (perpetuidad sin crecimiento) y la brecha frente
al precio de mercado, por emisor y por año.

MODELO USADO Y POR QUÉ (decisiones que debes poder explicar en la
disertación)
-------------------------------------------------------------------------
1. PERPETUIDAD SIN CRECIMIENTO (no Gordon-Shapiro):
       Valor_teorico = Dividendo_anual / Ke
   Se eligió esta versión, y no la perpetuidad creciente, porque el tema
   asignado habla de "dividendo ESTABLE", no de dividendo creciente, y
   porque no se cuenta con una tasa de crecimiento (g) estimada con
   evidencia propia. Usar "g" sin sustento sería inventar un dato.

2. Ke POR CAPM, CON BETA PROPIO Y ERP DE FUENTE CITADA:
       Ke = Rf + Beta x ERP_Peru
   - Rf: tasa libre de riesgo diaria, de ^TNX (ya calculada en 03).
   - Beta: el que calculó 03_limpieza_datos.py por regresión propia
     (retorno del emisor vs. retorno de EPU).
   - ERP_Peru = 6.30%: Prima de Riesgo de Mercado TOTAL para Perú
     (incluye la prima madura de EE.UU. + el riesgo país), tomada de:
     Damodaran, A. (2026). Country Default Spreads and Risk Premiums
     (pagina actualizada al 5 de enero de 2026, consultada 24-09-2026).
     https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ctryprem.html
     Peru: default spread ajustado 1.36% | Prima de riesgo pais 2.07% |
     ERP TOTAL 6.30% | Calificacion Baa1.
   Esta pagina se actualiza dos veces al ano (enero y julio): la cifra
   citada aqui corresponde a la version vigente al momento de escribir
   este script, y debe verificarse de nuevo si pasa mucho tiempo antes
   de la entrega final.
   Esta cifra es un PARÁMETRO PUBLICADO, no un dato extraído de una API:
   así se trabaja en toda la literatura de valuación (no se "descarga",
   se cita).

3. LIMITACIÓN DE DISEÑO QUE DEBES DECLARAR EN TU ARTÍCULO (no se oculta):
   Rf y ERP_Peru están pensados para combinarse con flujos en DÓLARES.
   El Ke resultante es, en rigor, una tasa en dólares, mientras que 8 de
   los 10 emisores pagan dividendo en SOLES (Dividendo_efectivo). Este
   descalce de moneda se aplica igual a los 10 emisores por simplicidad
   y se declara aquí como limitación metodológica explícita. Corregirlo
   (vía tipo de cambio o relación de Fisher) queda fuera del alcance de
   este script.

4. AGREGACIÓN ANUAL, NO DIARIA: el dividendo se paga distinto número de
   veces al año según el emisor (ver 03: UNACEM tuvo ~30 pagos en 8 años,
   Alicorp ~9). Por eso el resultado se arma por año calendario: se suma
   el dividendo pagado ese año, se toma el precio de cierre de fin de año
   (el último Close disponible) y el Rf promedio de ese año para ese
   emisor.

5. EVENTOS DE DIVIDENDO ATIPICO DOCUMENTADOS (2021): tres emisores
   (Alicorp, Backus, Cementos Pacasmayo) pagaron en 2021 un dividendo que,
   segun sus propios comunicados oficiales (hechos de importancia / SEC
   Form 6-K), correspondia a utilidades represadas de anos anteriores
   (2014-2019 en el caso de Pacasmayo), no al ejercicio 2021. Esto rompe
   el supuesto de "dividendo estable" del modelo. Se marcan explicitamente
   en EVENTOS_DIVIDENDO_ATIPICO, CON FUENTE CITADA, y se reporta la prueba
   estadistica dos veces: con la muestra completa y como analisis de
   robustez excluyendo estos 3 casos documentados. No se ocultan datos:
   ambos resultados quedan en el archivo de salida.

Instalación:  pip install pandas numpy scipy matplotlib
Ejecución:    python 04_analisis.py   (ejecutar desde /codigo)
Salidas:      ../salidas/tabla_valuacion_<CODIGO>.csv
              ../salidas/prueba_brecha_<CODIGO>.txt
              ../salidas/eventos_dividendo_atipico_<CODIGO>.csv
              ../salidas/fig_valor_teorico_vs_precio_<CODIGO>.png
              ../log_ejecucion.txt (se agrega una línea)
"""

import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. RUTAS (mismo patrón que 01, 02 y 03)
# ----------------------------------------------------------------------------
CODIGO_MATRICULA = "2024200514G"

try:
    DIR_CODIGO = Path(__file__).resolve().parent
    DIR_PROYECTO = DIR_CODIGO.parent
except NameError:
    DIR_ACTUAL = Path.cwd()
    DIR_PROYECTO = DIR_ACTUAL.parent if DIR_ACTUAL.name == "codigo" else DIR_ACTUAL

DIR_PROCESADOS = DIR_PROYECTO / "datos_procesados"
DIR_SALIDAS = DIR_PROYECTO / "salidas"
LOG_PATH = DIR_PROYECTO / "log_ejecucion.txt"
DIR_SALIDAS.mkdir(parents=True, exist_ok=True)

RUTA_PANEL = DIR_PROCESADOS / f"datos_procesados_{CODIGO_MATRICULA}.csv"
RUTA_BETA = DIR_PROCESADOS / f"beta_emisores_{CODIGO_MATRICULA}.csv"

# ----------------------------------------------------------------------------
# 1. PARÁMETRO CITADO (no se calcula, se declara y se referencia)
# ----------------------------------------------------------------------------
ERP_PERU = 0.0630   # Damodaran (2026), dataset "Country Default Spreads and
                     # Risk Premiums", actualizado 05-ene-2026. Ver docstring.
FUENTE_ERP = ("Damodaran, A. (2026). Country Default Spreads and Risk "
              "Premiums. https://pages.stern.nyu.edu/~adamodar/New_Home_Page/"
              "datafile/ctryprem.html (Peru, calificacion Baa1, ERP total = "
              "6.30%, pagina actualizada 05-01-2026, consultada 24-09-2026)")

# Eventos de dividendo atipico DOCUMENTADOS con fuente primaria. No es un
# criterio estadistico (tipo "excluir outliers"): son casos verificados de
# que el pago de ese ano-emisor no correspondio a un dividendo regular del
# ejercicio, sino a utilidades represadas de anos anteriores, liberadas de
# golpe. Esto rompe el supuesto de "dividendo estable" del modelo. El
# criterio se declara ANTES de ver los resultados de la prueba, evitando
# que la exclusion parezca un ajuste posterior al resultado (numeral 2.4.6
# y Criterio 6 de la rubrica: etica academica).
EVENTOS_DIVIDENDO_ATIPICO = {
    ("ALICORC1.LM", 2021): {
        "motivo": "Dividendo extraordinario de S/ 0.585/accion aprobado el "
                  "26-jul-2021, adicional al reparto regular. El total "
                  "distribuido en 2021 (S/713.6 MM) triplico el de 2020 y "
                  "2022 (~S/213.6 MM cada uno).",
        "fuente": "Alicorp S.A.A. (2021). Resultados 2T21. "
                  "https://www.alicorp.com.pe/media/conference_calls/"
                  "Alicorp_Earnings_Report_2Q21_ES_VF.pdf",
    },
    ("BACKUSI1.LM", 2021): {
        "motivo": "La empresa no pago dividendo ordinario desde marzo de "
                  "2018; el pago del 25-jun-2021 correspondio a utilidades "
                  "represadas. Un Hecho de Importancia posterior (05-dic-"
                  "2025) confirma que el Directorio aun distribuia, casi 4 "
                  "anos despues, el remanente de utilidades del ejercicio "
                  "2021 (S/2,935.14 MM en conjunto con 2022), evidenciando "
                  "un patron de distribucion irregular y prolongado. El PDF "
                  "original del Hecho de Importancia en smv.gob.pe no fue "
                  "accesible al momento de la consulta (portal con bloqueo "
                  "de solicitudes automatizadas, ver incidencias_fuente.md); "
                  "se cita el reporte periodistico que transcribe su "
                  "contenido.",
        "fuente": "Redaccion Gestion. (2025, 10 de diciembre). Backus "
                  "repartira S/ 2,935 millones en utilidades a sus "
                  "accionistas mayoritarios: ¿quienes son? Gestion. "
                  "https://gestion.pe/economia/empresas/backus-repartira-"
                  "s-2935-millones-en-utilidades-a-sus-accionistas-"
                  "mayoritarios-quienes-son-noticia/",
    },
    ("CPACASC1.LM", 2021): {
        "motivo": "Dividendo de S/0.79/accion (~S/366.7 MM) pagado en 2021, "
                  "declarado explicitamente por la empresa como proveniente "
                  "de resultados acumulados de los ejercicios 2014 a 2019, "
                  "no de las utilidades de 2021.",
        "fuente": "Cementos Pacasmayo S.A.A. (2021). Form 6-K, SEC. "
                  "https://www.sec.gov/Archives/edgar/data/1221029/"
                  "000121390021017459/ea138327ex99-1_cementospacas.htm",
    },
}


def registrar_log(lineas):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        for linea in lineas:
            f.write(linea + "\n")


def main():
    print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")
    print(f"ERP Perú usado: {ERP_PERU:.4f} | Fuente: {FUENTE_ERP}\n")

    lineas_log = [
        "=" * 78,
        f"Corrida (análisis): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} "
        f"| Código de matrícula: {CODIGO_MATRICULA}",
        f"ERP Perú: {ERP_PERU} | Fuente: {FUENTE_ERP}",
    ]

    if not RUTA_PANEL.exists() or not RUTA_BETA.exists():
        print("ERROR: falta datos_procesados o beta_emisores. Corre primero "
              "03_limpieza_datos.py.")
        lineas_log.append("Resultado: FALLO | archivos de entrada no encontrados")
        registrar_log(lineas_log)
        return

    panel = pd.read_csv(RUTA_PANEL, parse_dates=["Date"])
    beta_df = pd.read_csv(RUTA_BETA)
    print(f"Panel leído: {len(panel)} filas | {panel['Ticker'].nunique()} emisores")

    # ---- 2. Agregación anual por emisor ----
    panel["Anio"] = panel["Date"].dt.year

    agg = (
        panel.groupby(["Ticker", "Nemonico_BVL", "Emisor", "Anio"])
        .agg(
            Dividendo_anual=("Dividendo_efectivo", "sum"),
            Precio_cierre_fin_anio=("Close", "last"),
            Rf_promedio_anual=("Rf", "mean"),
            Obs_anio=("Close", "size"),
        )
        .reset_index()
    )

    # ---- 3. Unir con el beta de cada emisor (constante en el tiempo) ----
    tabla = agg.merge(
        beta_df[["Ticker", "beta", "r2", "p_valor"]], on="Ticker", how="left"
    )

    sin_beta = tabla["beta"].isna().sum()
    if sin_beta:
        print(f"AVISO: {sin_beta} filas sin beta disponible; se excluyen "
              f"del cálculo de Ke y valor teórico.")
    lineas_log.append(f"Filas sin beta: {sin_beta}")

    # ---- 4. Ke (CAPM) y valor teórico (perpetuidad sin crecimiento) ----
    tabla["Ke"] = tabla["Rf_promedio_anual"] + tabla["beta"] * ERP_PERU

    # Salvaguarda: el modelo de perpetuidad exige Ke > 0. Si algún emisor
    # sale con Ke <= 0 (posible con beta negativo y Rf bajo), NO se calcula
    # un valor teórico sin sentido: se deja como NaN y se avisa, en vez de
    # dividir por un número negativo o cercano a cero en silencio.
    tabla["Ke_valida"] = tabla["Ke"] > 0
    tabla["Valor_teorico"] = np.where(
        tabla["Ke_valida"], tabla["Dividendo_anual"] / tabla["Ke"], np.nan
    )

    n_ke_invalida = (~tabla["Ke_valida"]).sum()
    if n_ke_invalida:
        print(f"AVISO: {n_ke_invalida} fila(s) con Ke <= 0 (beta negativo "
              f"combinado con Rf bajo). Se dejan sin valor teórico en vez "
              f"de calcular un resultado sin sentido económico. Revísalas "
              f"en la tabla antes de interpretar los resultados.")
    lineas_log.append(f"Filas con Ke invalida (<=0): {n_ke_invalida}")

    # Años sin ningún dividendo pagado: el valor teórico da 0 por
    # construcción (perpetuidad de un dividendo nulo es 0). Se marca
    # explícitamente para no confundirlo con un error de cálculo.
    tabla["Sin_dividendo_ese_anio"] = tabla["Dividendo_anual"] == 0

    # ---- 5. Brecha frente al precio de mercado ----
    tabla["Brecha"] = tabla["Precio_cierre_fin_anio"] - tabla["Valor_teorico"]
    tabla["Brecha_pct"] = tabla["Brecha"] / tabla["Precio_cierre_fin_anio"]

    # ---- 5.1 Marcar eventos de dividendo atipico documentados ----
    tabla["Evento_dividendo_atipico"] = tabla.apply(
        lambda r: (r["Ticker"], r["Anio"]) in EVENTOS_DIVIDENDO_ATIPICO, axis=1
    )
    tabla["Nota_evento_atipico"] = tabla.apply(
        lambda r: EVENTOS_DIVIDENDO_ATIPICO.get(
            (r["Ticker"], r["Anio"]), {}
        ).get("motivo", ""),
        axis=1,
    )

    tabla = tabla.sort_values(["Ticker", "Anio"])
    columnas_finales = [
        "Ticker", "Nemonico_BVL", "Emisor", "Anio",
        "Dividendo_anual", "Sin_dividendo_ese_anio",
        "beta", "Rf_promedio_anual", "Ke", "Ke_valida",
        "Valor_teorico", "Precio_cierre_fin_anio",
        "Brecha", "Brecha_pct", "Evento_dividendo_atipico",
        "Nota_evento_atipico", "r2", "p_valor", "Obs_anio",
    ]
    tabla_final = tabla[columnas_finales]

    ruta_tabla = DIR_SALIDAS / f"tabla_valuacion_{CODIGO_MATRICULA}.csv"
    tabla_final.to_csv(ruta_tabla, index=False, encoding="utf-8-sig")
    print(f"\nGuardado: {ruta_tabla} | {len(tabla_final)} filas "
          f"(esperado: 10 emisores x 8 años = 80)")

    # ---- 6. Prueba estadistica sobre la brecha: muestra completa y
    #         analisis de robustez excluyendo eventos documentados ----
    base_valida = tabla_final[
        (~tabla_final["Sin_dividendo_ese_anio"]) & (tabla_final["Ke_valida"])
    ]
    brecha_completa = base_valida["Brecha_pct"].dropna()
    brecha_robustez = base_valida.loc[
        ~base_valida["Evento_dividendo_atipico"], "Brecha_pct"
    ].dropna()

    def _resumen_prueba(serie, etiqueta):
        t_stat, p_t = stats.ttest_1samp(serie, popmean=0)
        try:
            w_stat, p_w = stats.wilcoxon(serie)
            linea_w = (f"Prueba de Wilcoxon: W={w_stat:.4f} | "
                       f"p-valor={p_w:.6f}")
        except Exception as e:
            p_w = None
            linea_w = f"Wilcoxon no se pudo calcular: {e}"
        interp = (
            "p-valor < 0.05: se rechaza H0, la brecha promedio es "
            "estadisticamente distinta de cero."
            if p_t < 0.05 else
            "p-valor >= 0.05: no se rechaza H0, no hay evidencia de que "
            "la brecha promedio sea distinta de cero."
        )
        return [
            f"--- {etiqueta} ---",
            f"Observaciones: {len(serie)}",
            f"Brecha promedio: {serie.mean():.4f} ({serie.mean()*100:.2f}%)",
            f"Desviacion estandar: {serie.std():.4f}",
            f"Prueba t (H0: brecha promedio = 0): t={t_stat:.4f} | "
            f"p-valor={p_t:.6f}",
            linea_w,
            f"Interpretacion: {interp}",
            "",
        ], p_t

    lineas_prueba = [
        "PRUEBA SOBRE LA BRECHA (Precio de mercado - Valor teorico) / Precio",
        "",
    ]
    bloque_completo, p_t_completo = _resumen_prueba(
        brecha_completa, "MUESTRA COMPLETA (79 obs, sin filtrar eventos)"
    )
    lineas_prueba += bloque_completo

    lineas_prueba.append(
        "EVENTOS DE DIVIDENDO ATIPICO EXCLUIDOS EN EL ANALISIS DE ROBUSTEZ "
        "(documentados con fuente primaria, ver detalle debajo):"
    )
    for (ticker, anio), info in EVENTOS_DIVIDENDO_ATIPICO.items():
        lineas_prueba.append(f"  - {ticker} ({anio}): {info['motivo']}")
        lineas_prueba.append(f"    Fuente: {info['fuente']}")
    lineas_prueba.append("")

    bloque_robustez, p_t_robustez = _resumen_prueba(
        brecha_robustez,
        "ANALISIS DE ROBUSTEZ (excluyendo los 3 eventos documentados)",
    )
    lineas_prueba += bloque_robustez

    ruta_prueba = DIR_SALIDAS / f"prueba_brecha_{CODIGO_MATRICULA}.txt"
    with open(ruta_prueba, "w", encoding="utf-8") as f:
        f.write("\n".join(lineas_prueba) + "\n")
    print(f"Guardado: {ruta_prueba}")
    print("\n".join(lineas_prueba))

    # Tabla aparte de los eventos documentados, lista para usar como anexo
    # o tabla de resultados en el articulo (mas facil de citar que el .txt)
    filas_eventos = []
    for (ticker, anio), info in EVENTOS_DIVIDENDO_ATIPICO.items():
        filas_eventos.append({
            "Ticker": ticker, "Anio": anio,
            "Motivo": info["motivo"], "Fuente": info["fuente"],
        })
    eventos_df = pd.DataFrame(filas_eventos)
    ruta_eventos = DIR_SALIDAS / f"eventos_dividendo_atipico_{CODIGO_MATRICULA}.csv"
    eventos_df.to_csv(ruta_eventos, index=False, encoding="utf-8-sig")
    print(f"Guardado: {ruta_eventos}")

    brecha_valida = brecha_completa  # se usa mas abajo en el log final

    # ---- 7. Figura: valor teorico vs. precio de mercado, por emisor ----
    tickers = tabla_final["Ticker"].unique()
    fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharex=True)
    for ax, ticker in zip(axes.flat, tickers):
        datos = tabla_final[tabla_final["Ticker"] == ticker]
        ax.plot(datos["Anio"], datos["Precio_cierre_fin_anio"],
                marker="o", label="Precio de mercado", color="steelblue")
        ax.plot(datos["Anio"], datos["Valor_teorico"],
                marker="s", label="Valor teorico", color="firebrick")
        ax.set_title(ticker, fontsize=9)
        ax.tick_params(labelsize=7)
    axes.flat[0].legend(fontsize=7, loc="best")
    fig.suptitle("Valor teorico (perpetuidad) vs. precio de mercado, por emisor")
    fig.tight_layout()
    ruta_fig = DIR_SALIDAS / f"fig_valor_teorico_vs_precio_{CODIGO_MATRICULA}.png"
    fig.savefig(ruta_fig, dpi=120)
    print(f"Guardado: {ruta_fig}")
    plt.close(fig)

    # ---- 8. Cerrar el log ----
    lineas_log.append(
        f"Resultado: EXITO | tabla: {len(tabla_final)} filas | "
        f"brecha promedio (completa): {brecha_completa.mean():.4f} | "
        f"p-valor t (completa): {p_t_completo:.6f} | "
        f"brecha promedio (robustez): {brecha_robustez.mean():.4f} | "
        f"p-valor t (robustez): {p_t_robustez:.6f}"
    )
    registrar_log(lineas_log)
    print(f"\nLog actualizado en: {LOG_PATH}")


if __name__ == "__main__":
    main()

Python 3.13.15 | pandas 2.2.3
ERP Perú usado: 0.0630 | Fuente: Damodaran, A. (2026). Country Default Spreads and Risk Premiums. https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ctryprem.html (Peru, calificacion Baa1, ERP total = 6.30%, pagina actualizada 05-01-2026, consultada 24-09-2026)

Panel leído: 20037 filas | 10 emisores

Guardado: /content/salidas/tabla_valuacion_2024200514G.csv | 80 filas (esperado: 10 emisores x 8 años = 80)
Guardado: /content/salidas/prueba_brecha_2024200514G.txt
PRUEBA SOBRE LA BRECHA (Precio de mercado - Valor teorico) / Precio

--- MUESTRA COMPLETA (79 obs, sin filtrar eventos) ---
Observaciones: 79
Brecha promedio: -0.2770 (-27.70%)
Desviacion estandar: 1.1311
Prueba t (H0: brecha promedio = 0): t=-2.1770 | p-valor=0.032506
Prueba de Wilcoxon: W=1329.0000 | p-valor=0.219952
Interpretacion: p-valor < 0.05: se rechaza H0, la brecha promedio es estadisticamente distinta de cero.

EVENTOS DE DIVIDENDO ATIPICO EXCLUIDOS EN EL ANALISIS DE ROBUSTEZ 